# Project - Airline AI Assistant

### Extends the basic airline assistant (Week 2, day 4) with a tool to set ticket prices. Uses a `FlightAIAssistant` class

**Features**
- Get ticket prices
- Set ticket prices
- Gradio interface
- local db




**Import packages and initialize openai client and sqlite db**

In [ ]:
# Import packages
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [ ]:
# Initialization
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()
DB = "flight_prices.db"

**Define the FlightTools class with function specifications for the assistant to use**

In [ ]:
class FlightTools:
    @staticmethod
    def get_price_function():
        return {
            "name": "get_ticket_price",
            "description": "Get the price of a return ticket to the destination city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "destination_city": {
                        "type": "string",
                        "description": "The city that the customer wants to travel to",
                    },
                },
                "required": ["destination_city"],
                "additionalProperties": False
            }
        }
        
    @staticmethod
    def set_price_function():
        return {
            "name": "set_ticket_price",
            "description": "Set the price of a return ticket to the destination city.",
            "parameters": {
                "type": "object",
            "properties": {
                "destination_city": {
                    "type": "string",
                    "description": "The city that the customer wants to travel to",
                },
                "price": {
                    "type": "string",
                    "description": "The price of the ticket" 
                }
            },
            "required": ["destination_city", "price"],
            "additionalProperties": False
        }
    }


**Define the FlightAIAssistant class**

In [ ]:
class FlightAIAssistant:
    system_message = """
        You are a helpful assistant for an Airline called FlightAI.
        Give short, courteous answers, no more than 1 sentence.
        Always be accurate. If you don't know the answer, say so.
        Don't make up answers. If the user asks for something you can't do, politely decline.
    """
    
    def __init__(self, db_path, openai_client, model):
        self.db_path = db_path
        self.openai = openai_client
        self.model = model
        self.tools = [
            {"type": "function", "function": self.get_price_function()},
            {"type": "function", "function": self.set_price_function()}
        ]
        self.system_message = self.system_message

    @staticmethod
    def get_price_function():
        return FlightTools.get_price_function()

    @staticmethod
    def set_price_function():
        return FlightTools.set_price_function()

    def get_ticket_price(self, city):
        with sqlite3.connect(self.db_path) as conn:
            cursor = conn.cursor()
            cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
            result = cursor.fetchone()
            return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

    def set_ticket_price(self, city, price):
        with sqlite3.connect(self.db_path) as conn:
            cursor = conn.cursor()
            cursor.execute(
                'INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?',
                (city.lower(), price, price)
            )
            conn.commit()

    def handle_tool_calls(self, message):
        responses = []
        tool_handlers = {
            "get_ticket_price": self._handle_get_ticket_price,
            "set_ticket_price": self._handle_set_ticket_price,
            # Add more tool as needed
        }

        for tool_call in message.tool_calls:
            arguments = json.loads(tool_call.function.arguments)
            handler = tool_handlers.get(tool_call.function.name)
            if handler:
                responses.append(handler(tool_call, arguments))
            else:
                responses.append({
                    "role": "tool",
                    "content": f"Tool '{tool_call.function.name}' is not supported.",
                    "tool_call_id": tool_call.id
                })
        return responses

    def _handle_get_ticket_price(self, tool_call, arguments):
        city = arguments.get('destination_city')
        price_details = self.get_ticket_price(city)
        return {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }

    def _handle_set_ticket_price(self, tool_call, arguments):
        city = arguments.get('destination_city')
        price = arguments.get('price')
        self.set_ticket_price(city, price)
        return {
            "role": "tool",
            "content": f"Ticket price for {city} set to {price}",
            "tool_call_id": tool_call.id
        }

    def chat(self, message, history):
        history = [{"role": h["role"], "content": h["content"]} for h in history]
        messages = [{"role": "system", "content": self.system_message}] + history + [{"role": "user", "content": message}]
        response = self.openai.chat.completions.create(model=self.model, messages=messages, tools=self.tools)
        while response.choices[0].finish_reason == "tool_calls":
            message_obj = response.choices[0].message
            responses = self.handle_tool_calls(message_obj)
            messages.append(message_obj)
            messages.extend(responses)
            response = self.openai.chat.completions.create(model=self.model, messages=messages, tools=self.tools)
        return response.choices[0].message.content

    def launch_interface(self):
        gr.ChatInterface(fn=self.chat, type="messages").launch()
        
    def load_sample_data(self):
        sample_data = [
            ('london', 799),
            ('paris', 899),
            ('tokyo', 1420),
            ('sydney', 2999)
        ]
        with sqlite3.connect(DB) as conn:
            cursor = conn.cursor()
            cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price INTEGER)')
            cursor.executemany('INSERT OR REPLACE INTO prices (city, price) VALUES (?, ?)', sample_data)
            conn.commit()
            

**Launch AI assistant**

In [ ]:
ai_assistant = FlightAIAssistant(db_path=DB, openai_client=openai, model=MODEL)
ai_assistant.load_sample_data()
ai_assistant.launch_interface()